# ScholarAI v3 SOTA: Multi-GPU Backend Runner
This notebook orchestrates the fine-tuning and deployment of the **Deep Semantic Evasion** engine. 

### ⚠️ Prerequisites
1. **Internet**: Must be "ON" in the sidebar.
2. **Accelerator**: Must be "GPU T4 x2".
3. **Secrets**: Click the **Key icon** and add:
   - `HF_TOKEN`: Your HuggingFace token.
   - `NGROK_TOKEN`: Your ngrok auth token.

In [ ]:
# --- STEP 1: INSTALL DEPENDENCIES ---
!pip install --upgrade amrlib fastapi uvicorn "pydantic<=2.12.3" accelerate bitsandbytes trl peft datasets \
    python-multipart pyngrok penman unidecode huggingface_hub sentencepiece protobuf \
    word2number celery redis python-jose[cryptography] passlib[bcrypt] slowapi \
    "numpy<2.1" "pillow<12.0"
!pip install git+https://github.com/huggingface/transformers.git

In [ ]:
# --- STEP 2: REPOSITORY SYNC ---
import os
%cd /kaggle/working

if os.path.exists("/kaggle/working/sensorspine-humaniser-v3"):
    print("🔄 Updating existing repository...")
    %cd /kaggle/working/sensorspine-humaniser-v3
    !git reset --hard HEAD
    !git pull origin main
else:
    print("📥 Cloning fresh repository...")
    !git clone https://github.com/NandishSinha1403/sensorspine-humaniser-v3.git
    %cd sensorspine-humaniser-v3

print("✅ Repository is ready.")

In [ ]:
# --- STEP 3: AUTH & ENVIRONMENT SETUP ---
from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok
from huggingface_hub import login

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
NGROK_TOKEN = user_secrets.get_secret("NGROK_TOKEN")

login(token=HF_TOKEN)
ngrok.set_auth_token(NGROK_TOKEN)

print("✅ Environment initialized for 2x T4 GPUs.")

### Phase 3: CLM Fine-Tuning (2x T4 DDP)
Uses `torchrun` to utilize both GPUs and the new stability fixes.

In [ ]:
# Launch with Distributed Data Parallel (DDP)
!torchrun --nproc_per_node=2 fine_tune_clm.py

### Phase 4: DPO Penalization (Decoupled Generation + 2x T4 DDP)
Uses the DiagnosticJudge to penalize AI-like patterns. Generation is done sequentially to prevent DDP timeouts.

In [ ]:
# Step 1: Generate DPO Dataset (Single GPU to prevent DDP timeout)
!python generate_dpo_data.py

# Step 2: DPO Training (2x T4 DDP)
!torchrun --nproc_per_node=2 fine_tune_dpo.py

### Phase 5: Deployment (FastAPI + ngrok)
Starts the backend server and creates a public tunnel.

In [ ]:
ngrok.kill()
public_url = ngrok.connect(8000).public_url
print(f"🚀 PUBLIC URL: {public_url}")

%cd backend
!python main.py